# 6. Concepts as features

If NAVE's concepts capture what matters, the share of each concept in an image should be enough to
classify it. The paper (Section 4.2, Table 1) extracts K = 20 concepts per class on the training set,
merges them, and trains a random forest on the concept proportions.

Here is the same protocol on a toy two-class problem built from one photo: crops from its upper
part (flag, stars, face) against crops from its lower part (uniform, medals).

In [ ]:
import sys
if "google.colab" in sys.modules:       # on Google Colab: install what this notebook needs
    get_ipython().run_line_magic("pip", 'install -q "nave-xai"')
# elsewhere: pip install nave-xai
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cbook
from matplotlib.image import imread
from nave_xai import NAVEImage

image = imread(cbook.get_sample_data("grace_hopper.jpg", asfileobj=False))   # ships with matplotlib

def show(ax, explainer, exp, title=""):
    """An explanation map in the explainer's concept colours."""
    ax.imshow(explainer.palette[exp.explanation]); ax.set_title(title, fontsize=10); ax.axis("off")

TRAINING = "imagenet"     # "random" to skip downloads

In [ ]:
rng = np.random.default_rng(0)

def crops(img, n, size=128):
    H, W = img.shape[:2]
    out = []
    for _ in range(n):
        y, x = rng.integers(0, H - size), rng.integers(0, W - size)
        c = img[y:y + size, x:x + size]
        out.append(c[:, ::-1] if rng.random() < 0.5 else c)
    return out

X_img = crops(image[:280], 40) + crops(image[320:], 40)
y = np.array([0] * 40 + [1] * 40)
order = rng.permutation(len(y))
train, test = order[:50], order[50:]

Concepts are extracted per class (`groups=`), as in the paper; every image then becomes the vector of
its concept proportions.

In [ ]:
explainer = NAVEImage(model="resnet50", training=TRAINING, k=10).fit([X_img[i] for i in train], groups=y[train])
K = len(explainer.palette)
print(K, "concepts in total; classes", explainer.classes)

def proportions(images):
    return np.stack([np.bincount(e.explanation.ravel(), minlength=K) / e.explanation.size
                     for e in explainer.predict(images)])

P_train = proportions([X_img[i] for i in train])
P_test = proportions([X_img[i] for i in test])

In [ ]:
try:
    from sklearn.ensemble import RandomForestClassifier
    clf = RandomForestClassifier(200, random_state=0).fit(P_train, y[train])
    print(f"random forest on concept proportions: {100 * (clf.predict(P_test) == y[test]).mean():.0f}% test accuracy")
except ImportError:                                   # without scikit-learn: nearest class mean
    means = np.stack([P_train[y[train] == c].mean(0) for c in (0, 1)])
    pred = ((P_test[:, None] - means[None]) ** 2).sum(2).argmin(1)
    print(f"nearest class mean on concept proportions: {100 * (pred == y[test]).mean():.0f}% test accuracy")

The correlation of every concept with the classes, computed by `fit`:

In [ ]:
plt.figure(figsize=(8, 1.8))
plt.imshow(explainer.correlation.T, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
plt.yticks([0, 1], ["upper", "lower"]); plt.xlabel("concept"); plt.colorbar();

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  editor    = {Ceci, Michelangelo and Kocev, Dragi and Monreale, Anna and Vens, Celine and Flach, Peter
               and Manco, Giuseppe and Spiliopoulou, Myra and Masciari, Elio},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26},
  isbn      = {978-3-032-37664-0}
}
```